In [2]:
!pip install xgboost

In [3]:
# 必要なライブラリをインポート
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib # モデルの保存に使用
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import os
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# --- 1. データの読み込み ---
features_data_path = '../data/processed/features.parquet'
df = pd.read_parquet(features_data_path)

In [5]:
# 1. クラスタリング結果を読み込み

cluster_all_df = pd.read_csv('prefecture_cluster_all_methods.csv', encoding='utf-8-sig')
print("\n✓ クラスタリング結果を読み込みました")
print(f"  利用可能なクラスタリング方法: {len(cluster_all_df.columns)-1}種類")

available_methods = [col for col in cluster_all_df.columns if col != '都道府県名']
for i, method in enumerate(available_methods, 1):
    print(f"  {i}. {method}")


✓ クラスタリング結果を読み込みました
  利用可能なクラスタリング方法: 4種類
  1. クラスタ_簡単版
  2. クラスタ_件数考慮
  3. クラスタ_階層
  4. クラスタ_全特徴


In [6]:
df.columns

Index(['市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '取引価格（総額）_log', '取引時点での築年数', '取引の事情等_その他事情有り',
       '取引の事情等_他の権利・負担付き', '取引の事情等_他の権利・負担付き、調停・競売等', '取引の事情等_瑕疵有りの可能性',
       '取引の事情等_調停・競売等', '取引の事情等_調停・競売等、瑕疵有りの可能性', '取引の事情等_関係者間取引',
       '取引の事情等_関係者間取引、瑕疵有りの可能性', '取引の事情等_関係者間取引、調停・競売等', '取引の事情等_その他',
       '改装_改装済', '改装_未改装', '間取り_grouped_その他', '間取り_grouped_オープンフロア',
       '間取り_grouped_欠損値', '間取り_grouped_１ＤＫ', '間取り_grouped_１Ｋ',
       '間取り_grouped_１ＬＤＫ', '間取り_grouped_１Ｒ', '間取り_grouped_２ＤＫ',
       '間取り_grouped_２Ｋ', '間取り_grouped_２ＬＤＫ', '間取り_grouped_２ＬＤＫ＋Ｓ',
       '間取り_grouped_３ＤＫ', '間取り_grouped_３ＬＤＫ', '間取り_grouped_４ＤＫ',
       '間取り_grouped_４ＬＤＫ', '築年数_2乗', '築年数_3乗', '築年数_log', '面積_log', '面積_平方根',
       '築年数×面積', '築年数×駅距離', '築年数×建ぺい率', '築年数×容積率', '人口密度', '市区町村人口密度',
       '築年数×人口密度', '面積×駅距離', '面積×建ぺい率', '面積×容積率', '面積×人口密度', '建築可能性',
       '容積率_建ぺい率比', '建ぺい率_2乗'

In [16]:

# --- 説明変数・目的変数のセット ---
drop_cols = ['市区町村コード', 
             '間取り','用途', '今後の利用目的', 
             '都市計画','取引時点', '取引価格（総額）_log', '取引の事情等_その他',
             '改装_未改装', '間取り_grouped_その他'
             # '市区町村_頻度', '駅名_頻度', '地区名_頻度'
             ,'クラスタ']

X_cols = [col for col in df.columns if col not in drop_cols]
print(f"\n使用する特徴量の数: {len(X_cols)}個")

X = df[X_cols].copy()
Y = df['取引価格（総額）_log'].copy()


使用する特徴量の数: 73個


In [17]:
# カテゴリカラムを 'category' 型に変換
categorical_cols = ['都道府県名', '市区町村名', '地区名', '最寄駅：名称', '建物の構造']

for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].astype('category')

# 数値カラムのみクリーニング
numerical_cols = X.select_dtypes(include=[np.number]).columns
X[numerical_cols] = X[numerical_cols].replace([np.inf, -np.inf], np.nan)
X[numerical_cols] = X[numerical_cols].fillna(X[numerical_cols].median())

In [18]:
##ベースラインモデル（クラスタリングなし）
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

model_base = XGBRegressor(
    enable_categorical=True,
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    missing=np.nan
)

model_base.fit(
    X_train_base, y_train_base,
    eval_set=[(X_test_base, y_test_base)]
)

y_pred_base = model_base.predict(X_test_base)
mae_base = mean_absolute_error(y_test_base, y_pred_base)

print(f"\n【ベースライン結果】")
print(f"MAE: {mae_base:.6f}")

[0]	validation_0-rmse:0.33756
[1]	validation_0-rmse:0.32585
[2]	validation_0-rmse:0.31481
[3]	validation_0-rmse:0.30422
[4]	validation_0-rmse:0.29435
[5]	validation_0-rmse:0.28546
[6]	validation_0-rmse:0.27673
[7]	validation_0-rmse:0.26854
[8]	validation_0-rmse:0.26073
[9]	validation_0-rmse:0.25355
[10]	validation_0-rmse:0.24670
[11]	validation_0-rmse:0.24028
[12]	validation_0-rmse:0.23427
[13]	validation_0-rmse:0.22866
[14]	validation_0-rmse:0.22363
[15]	validation_0-rmse:0.21880
[16]	validation_0-rmse:0.21397
[17]	validation_0-rmse:0.20976
[18]	validation_0-rmse:0.20561
[19]	validation_0-rmse:0.20171
[20]	validation_0-rmse:0.19808
[21]	validation_0-rmse:0.19457
[22]	validation_0-rmse:0.19165
[23]	validation_0-rmse:0.18874
[24]	validation_0-rmse:0.18586
[25]	validation_0-rmse:0.18308
[26]	validation_0-rmse:0.18064
[27]	validation_0-rmse:0.17837
[28]	validation_0-rmse:0.17630
[29]	validation_0-rmse:0.17424
[30]	validation_0-rmse:0.17227
[31]	validation_0-rmse:0.17050
[32]	validation_0-

In [19]:

# #クラスタリング方法選択バージョン

# ★★★ クラスタリング方法を選択 ★★★
SELECTED_METHOD = 'クラスタ_件数考慮'  # ← ここでどのクラスタリング方法を適用するか指定できる

# 選択肢: 'クラスタ_簡単版', 'クラスタ_件数考慮', 'クラスタ_階層', 'クラスタ_全特徴'

print(f"\n選択されたクラスタリング方法: {SELECTED_METHOD}")

# クラスタ情報を結合
cluster_mapping = cluster_all_df.set_index('都道府県名')[SELECTED_METHOD].to_dict()
df['クラスタ'] = df['都道府県名'].map(cluster_mapping)

print(f"\nクラスタの分布:")
for cluster_id in sorted(df['クラスタ'].unique()):
    count = (df['クラスタ'] == cluster_id).sum()
    print(f"  クラスタ {cluster_id}: {count:,}件")



選択されたクラスタリング方法: クラスタ_件数考慮

クラスタの分布:
  クラスタ 0: 166,503件
  クラスタ 1: 68,882件
  クラスタ 2: 316,260件


In [20]:
## クラスタごとにモデル構築

print(f"【{SELECTED_METHOD}】でクラスタごとにモデルを構築")

all_predictions = []
all_actuals = []
cluster_results = {}

for cluster_id in sorted(df['クラスタ'].unique()):
    print(f"\n{'='*80}")
    print(f"【クラスタ {cluster_id}】")
    print('='*80)
    
    cluster_mask = df['クラスタ'] == cluster_id
    X_cluster = X[cluster_mask]
    Y_cluster = Y[cluster_mask]
    
    cluster_prefs = sorted(df[cluster_mask]['都道府県名'].unique())
    print(f"対象都道府県 ({len(cluster_prefs)}個):")
    print(f"  {', '.join(cluster_prefs)}")
    print(f"データ件数: {len(X_cluster):,}件")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_cluster, Y_cluster, test_size=0.2, random_state=42
    )
    
    print(f"  訓練: {len(X_train):,}件 / テスト: {len(X_test):,}件")
    
    model_xgb = XGBRegressor(
        enable_categorical=True,
        objective='reg:squarederror',
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=50,
        missing=np.nan
    )
    
    print("\nモデル学習中...")
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)]
    )
    
    y_pred = model_xgb.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"\n【結果】")
    print(f"MAE: {mae:.6f}")
    
    all_predictions.extend(y_pred)
    all_actuals.extend(y_test)
    cluster_results[cluster_id] = {
        'mae': mae,
        'test_size': len(X_test)
    }


【クラスタ_件数考慮】でクラスタごとにモデルを構築

【クラスタ 0】
対象都道府県 (22個):
  三重県, 京都府, 佐賀県, 兵庫県, 千葉県, 和歌山県, 埼玉県, 奈良県, 宮城県, 山口県, 山形県, 岐阜県, 岡山県, 島根県, 広島県, 愛知県, 沖縄県, 滋賀県, 長崎県, 長野県, 高知県, 鳥取県
データ件数: 166,503件
  訓練: 133,202件 / テスト: 33,301件

モデル学習中...
[0]	validation_0-rmse:0.31095
[1]	validation_0-rmse:0.30072
[2]	validation_0-rmse:0.29111
[3]	validation_0-rmse:0.28206
[4]	validation_0-rmse:0.27362
[5]	validation_0-rmse:0.26617
[6]	validation_0-rmse:0.25852
[7]	validation_0-rmse:0.25147
[8]	validation_0-rmse:0.24477
[9]	validation_0-rmse:0.23859
[10]	validation_0-rmse:0.23270
[11]	validation_0-rmse:0.22727
[12]	validation_0-rmse:0.22208
[13]	validation_0-rmse:0.21719
[14]	validation_0-rmse:0.21278
[15]	validation_0-rmse:0.20863
[16]	validation_0-rmse:0.20466
[17]	validation_0-rmse:0.20092
[18]	validation_0-rmse:0.19734
[19]	validation_0-rmse:0.19403
[20]	validation_0-rmse:0.19103
[21]	validation_0-rmse:0.18807
[22]	validation_0-rmse:0.18548
[23]	validation_0-rmse:0.18297
[24]	validation_0-rmse:0.18053
[25]	validation_

In [22]:
## 最終結果(mae比較）

overall_mae = mean_absolute_error(all_actuals, all_predictions)

print(f"\n{SELECTED_METHOD}:")
print(f"  MAE: {overall_mae:.6f}")

print(f"\nベースライン（クラスタリングなし）:")
print(f"  MAE: {mae_base:.6f}")

improvement = (mae_base - overall_mae) / mae_base * 100
diff = mae_base - overall_mae

print("\n" + "="*80)
print(" 改善効果")
print("="*80)
print(f"改善率: {improvement:+.2f}%")
print(f"絶対差: {diff:+.6f}")


print("\nクラスタ別詳細:")
for cluster_id, result in cluster_results.items():
    print(f"  クラスタ {cluster_id}: MAE {result['mae']:.6f} ({result['test_size']:,}件)")



クラスタ_件数考慮:
  MAE: 0.076263

ベースライン（クラスタリングなし）:
  MAE: 0.077963

 改善効果
改善率: +2.18%
絶対差: +0.001700

クラスタ別詳細:
  クラスタ 0: MAE 0.081849 (33,301件)
  クラスタ 1: MAE 0.090698 (13,777件)
  クラスタ 2: MAE 0.070177 (63,252件)
